# Contrato Tecnico de Escopo - Fase 5.5

## 1. Metrica Primaria e Direcao no modo auto

- A definicao de metrica e direcao por tipo de tarefa fica formalizada por:
  - `_get_metric_key(task)`: `rmse` para regressao e `auc` para classificacao.
  - `_direction(task)`: `minimize` para regressao e `maximize` para classificacao.
- A mesma metrica primaria deve ser usada para:
  - objetivo do Optuna;
  - comparacao entre trials;
  - comparacao entre modelos no modo `auto`;
  - selecao do modelo final.

## 2. Escopo Minimo Obrigatorio do Relatorio Tecnico

- Formato: `HTML`
- Pasta de saida: `reports/` na raiz do projeto (criar automaticamente se nao existir)
- Nome: `report_YYYYMMDDhhmmss.html`

Secoes obrigatorias:

1. Parametros de execucao definidos pelo usuario
2. Volumetria total e divisao treino/teste
3. Numero de folds e percentual treino/teste
4. Media da variavel resposta em treino e teste
5. Estatisticas descritivas essenciais
6. Resumo da otimizacao Optuna (melhor trial, hiperparametros e curva)
7. Curva de aprendizado do melhor modelo (metrica x epoca/interacao)
8. Metricas de treino e teste do modelo escolhido
9. Top 20 covariaveis mais importantes
10. SHAP Top 20 com interpretacao de direcao e alerta de inversao de efeito

## 3. Dimensoes Temporal e Segmentativa (Evolutivo)

- Nesta fase, a analise temporal/segmentativa e condicional aos parametros disponiveis na interface.
- Se variavel temporal for fornecida, incluir estabilidade de volume e media da resposta ao longo do tempo.
- Se variaveis de segmento forem fornecidas, incluir cortes por segmento.
- Se nao forem fornecidas, o pipeline nao pode falhar e deve registrar a ausencia no relatorio.

## 4. Contrato de Artefatos para Reprodutibilidade e Deploy

Cada execucao deve gerar pacote unico em `artifacts/<run_id>/`.

Estrutura minima:

- `artifacts/<run_id>/model/`
- `artifacts/<run_id>/preprocess/`
- `artifacts/<run_id>/inference/`
- `artifacts/<run_id>/metadata/`
- `artifacts/<run_id>/reports/`

Arquivos obrigatorios:

- `artifacts/<run_id>/model/model.joblib` (ou equivalente para MLP)
- `artifacts/<run_id>/preprocess/pipeline.joblib`
- `artifacts/<run_id>/inference/predict.py`
- `artifacts/<run_id>/metadata/train_config.json`
- `artifacts/<run_id>/metadata/metrics_train.json`
- `artifacts/<run_id>/metadata/metrics_test.json`
- `artifacts/<run_id>/metadata/feature_schema.json`
- `artifacts/<run_id>/metadata/optuna_summary.json`
- `artifacts/<run_id>/metadata/versions.json`
- `artifacts/<run_id>/reports/report_YYYYMMDDhhmmss.html`

Ambiente de reproducao/deploy:

- `requirements.txt` e/ou `poetry.lock`
- comando de inferencia documentado
- estrutura pronta para containerizacao (Docker)

## 5. Interpretabilidade Condicionada a Tecnica

- LightGBM: feature importance + SHAP (top 20)
- MLP:
  - padrao: Permutation Importance (global) + ALE (direcao marginal)
  - SHAP opcional com amostragem/custo controlado

## 6. Criterios de Aceite

1. O modo `auto` seleciona o modelo com a metrica primaria correta por task e direcao correta.
2. O relatorio HTML e gerado em `reports/` com timestamp no nome.
3. O relatorio contem todas as secoes obrigatorias.
4. Ausencia de variavel temporal/segmentativa nao quebra o pipeline e e registrada no relatorio.
5. Todos os artefatos obrigatorios sao gerados em `artifacts/<run_id>/`.
6. O script de inferencia reutiliza o mesmo pipeline de transformacao do treino.
7. O conjunto de dependencias permite reproduzir treino e inferencia.

## 7. Observacao de Consistencia Tecnica

- Embora `_get_metric_key` e `_direction` estejam corretos, hoje ha risco funcional na classificacao:
  - `metricas_classificacao` espera `y_prob` para calcular AUC;
  - no fluxo atual, em alguns pontos AUC e chamado sem `y_prob`.
- Para o contrato ser cumprido de ponta a ponta, a inferencia de classificacao deve sempre registrar probabilidades (`predict_proba` no LightGBM e `sigmoid` no MLP) quando a metrica primaria for `auc`.


## Atualizacao de Implementacao - Contrato Cumprido (Interpretabilidade)

### Entrega tecnica implementada
- Threshold de classificacao alinhado entre modelagem e interpretabilidade
- LightGBM com importancia global + SHAP (dependente de disponibilidade de pacote)
- MLP com Permutation Importance + ALE
- Geracao de artefato versionado em `reports/interpretability_YYYYMMDDhhmmss.json`

### Observacoes
- SHAP para MLP permanece opcional/evolutivo por custo computacional
- Direcao de efeito e sinalizacao de inversao (via correlacao feature-shap vs feature-target) implementadas no bloco SHAP de LightGBM
